# Overture Maps Placesを使ってみる

## 1. 今回やること
DuckDBからOverture Maps Placesを読み、吉祥寺周辺のPOIを取得し、basic_category / taxonomyを確認して、カフェ相当を抽出します。比較・精度評価は行いません。

[実行方法・利用条件](../docs/overture.md) / [検証結果](../docs/overture-validation.md)

## 2. 準備
Python / uvとDuckDBを使用します。tokenやアカウントは不要です。ルートで `uv sync --locked --group notebooks` を実行してください。固定releaseは `2026-09-23.0`、schemaは `v2.0.0` です。

In [ ]:
from geoai_open_lab.overture import (
    connect_overture, load_aoi, fetch_overture_places,
    filter_overture_cafes, summarize_overture, save_overture_results,
)

## 3. 読み取り準備
httpfs / spatialを読み込み、公開GeoParquetを匿名で取得する準備をします。

In [ ]:
con = connect_overture()

## 4. 吉祥寺周辺のPOIを取得
Foursquare編と同じAOI設定を読みます。境界を含み、LIMITや営業状態フィルターは付けません。全件とはこのデータ・範囲内のレコードで、現実の店舗の網羅性を意味しません。

In [ ]:
kichijoji = load_aoi("configs/aoi/kichijoji.toml")
places = fetch_overture_places(con, bbox=kichijoji, release="2026-09-23.0")
places[["name", "latitude", "longitude"]].head()

## 5. basic_category / taxonomyを確認
basic_categoryは粗い分類、taxonomy.primaryは詳細分類、hierarchyは上位から詳細への経路です。alternatesはその主階層以外の追加分類です。旧categoriesは使いません。

In [ ]:
places[["basic_category", "taxonomy_primary", "taxonomy_hierarchy", "taxonomy_alternates"]].head()

## 6. カフェ相当を抽出
`basic_category IN ('cafe', 'coffee_shop')` で選びます。Overtureの分類上Internet Cafeや動物カフェ、Coffee Roasteryも含み得ます。店名検索やFoursquareとの共通分類ではありません。

In [ ]:
cafes = filter_overture_cafes(con)
cafes[["name", "taxonomy_primary", "operating_status"]].head(10)

In [ ]:
cafes.groupby("taxonomy_primary", dropna=False).size().sort_values(ascending=False)

## 7. 結果と注意点
閉店を含む営業状態は取得後に確認します。NULLを営業中とみなしません。confidenceは提供元による存在への確信度スコアで、独自閾値で評価しません。詳細な集計・実行計画・測定の限界はローカルmanifestと検証記録へ残します。

In [ ]:
print(f"AOI内POI: {len(places)}件 / カフェ相当: {len(cafes)}件")
summary = summarize_overture(con)
summary["operating_status"]

In [ ]:
result_dir = save_overture_results(con)
con.close()